In [11]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ---------------------------------------- 0.0/684.7 kB ? eta -:--:--
   ---------------------------------------- 684.7/684.7 kB 9.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 36.8 MB/s  0:00:00

   ---------------------------------------- 0/8 [flatbuffers]
   ----- ---------------------------------- 1/8 [wassima]
   ---------- ----------------------------- 2/8 [qh3]
   ---------- ----------------------------- 2/8 [qh3]
   -------------------- ------------------- 4/8 [jh2]
   -------------------- ------------------- 4/8 [jh2]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8

In [2]:
from datetime import datetime
import requests
import pandas as pd
import requests
import json

# --------------------------------
# 1. Greek Cities (Lat / Lon)
# --------------------------------
cities = [
    {"city": "Athens", "lat": 37.9838, "lon": 23.7275},
    {"city": "Thessaloniki", "lat": 40.6401, "lon": 22.9444},
    {"city": "Patras", "lat": 38.2466, "lon": 21.7346},
    {"city": "Heraklion", "lat": 35.3387, "lon": 25.1442},
    {"city": "Larissa", "lat": 39.6390, "lon": 22.4179}
]

# --------------------------------
# 2. City Name Transformation
# (English → Greek)
# --------------------------------
CITY_NAME_MAP = {
    "Athens": "Αθήνα",
    "Thessaloniki": "Θεσσαλονίκη",
    "Patras": "Πάτρα",
    "Heraklion": "Ηράκλειο",
    "Larissa": "Λάρισα"
}

# --------------------------------
# 3. WMO Weather Codes
# --------------------------------
WMO_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    61: "Rain",
    71: "Snow",
    80: "Rain showers",
    95: "Thunderstorm"
}

# --------------------------------
# 4. API Fetch Function
# --------------------------------
def fetch_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current_weather": True,
        "timezone": "UTC"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

# --------------------------------
# 5. Extract + Transform
# --------------------------------
records = []

for city in cities:
    raw = fetch_weather(city["lat"], city["lon"])
    current = raw["current_weather"]
    
    weather_code = current["weathercode"]

    record = {
        "city_en": city["city"],
        "city_gr": CITY_NAME_MAP.get(city["city"], city["city"]),
        "latitude": city["lat"],
        "longitude": city["lon"],
        "timestamp_utc": datetime.utcnow(),
        "temperature_c": current["temperature"],
        "wind_speed_kmh": current["windspeed"],
        "weather_code": weather_code,
        "weather_description": WMO_CODES.get(weather_code, "Unknown")
    }

    records.append(record)
    
df_weather = pd.DataFrame(records)

df_weather

,city_en,city_gr,latitude,longitude,timestamp_utc,temperature_c,wind_speed_kmh,weather_code,weather_description
0,Athens,Αθήνα,37.9838,23.7275,2026-01-10 10:31:03.320807,18.1,16.3,2,Partly cloudy
1,Thessaloniki,Θεσσαλονίκη,40.6401,22.9444,2026-01-10 10:31:03.490887,9.4,5.9,3,Overcast
2,Patras,Πάτρα,38.2466,21.7346,2026-01-10 10:31:03.657178,13.9,30.6,95,Thunderstorm
3,Heraklion,Ηράκλειο,35.3387,25.1442,2026-01-10 10:31:03.826032,19.5,19.9,1,Mainly clear
4,Larissa,Λάρισα,39.6390,22.4179,2026-01-10 10:31:03.994418,10.1,3.1,2,Partly cloudy
